# Phase D: Controller analysis

**Question:** did the controller fail to follow a reasonable plan, or did it faithfully carry out an unsafe one?

**How TF++'s controller works**, read from `model.control_pid_direct` and `nav_planner.get_throttle`:

- **Steering** is a PID controller that steers toward TF++'s planned path.
- **Throttle and brake** come from a fixed rule on the target speed and the current speed. **Brake** if the target speed is below 0.01 m/s, or speed / target > 1.1, or (when the car is not far below its target) speed / target > 1.031. **Otherwise** the throttle comes from a small linear formula.
- **Overrides** can change this: creeping forward when stuck, the LiDAR safety box that stops the creep, and the stop-sign brake.

| Section | Question |
|---|---|
| 1 | Does the logged throttle and brake match exactly what the rule says for the logged target speed? |
| 2 | **Speed tracking:** how far is the actual speed from the target speed? |
| 3 | **Path tracking:** where was the car 0.5 s and 1 s later, compared with the path planned at that moment? TF++ replans every step, so each plan is compared only with its own short future. |
| 4 | **Braking response:** once the brake comes on, how quickly does the car slow down? |
| 5 | Steering behaviour: saturation and reversals |
| 6 | One summary line per collision |

In [ ]:
import sys, csv, json, math, collections
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

REPO = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO / "src"))
import ew_analysis as ew
import pce_analysis as pce

RESULTS = REPO / "results" / "bench2drive_v2"
OUT = RESULTS / "pce"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "font.size": 9,
                     "axes.spines.top": False, "axes.spines.right": False})

def table(rows, cols=None, fmt=None):
    cols = cols or list(rows[0]); fmt = fmt or {}
    out = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for r in rows:
        out.append("| " + " | ".join(fmt[c](r[c]) if c in fmt and isinstance(r.get(c), (int, float)) and r.get(c) == r.get(c)
                                     else (f"{r[c]:.2f}" if isinstance(r.get(c), float) else str(r.get(c, ""))) for c in cols) + " |")
    display(Markdown("\n".join(out)))

def save_csv(name, rows):
    with open(OUT / f"{name}.csv", "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)

pct = lambda v: f"{100 * v:.0f}%"

# shared data: collision events and features from the early-warning cache, plans/poses from the pce cache
routes, ckpts, EW = ew.build_cache(RESULTS, REPO / "carla_garage/leaderboard/data/bench2drive220.xml", RESULTS / "early_warning" / "cache.pkl")
PC = pce.build_cache(RESULTS, OUT / "cache.pkl")
events = ew.event_table(routes, ckpts, EW)
CORE = ("junction", "lane_merge", "obstacle_detour")
crashes = [e for e in events if e["counted"] and e["clean_history"] and e["step"] >= 60]
for e in crashes:
    e["core"] = e["family"] in CORE
    e["ego_speed"] = e["v_pre"]          # ego speed 0.1 s before contact
print(f"collisions analysed: {len(crashes)} (counted by the leaderboard, no other contact in the 3 s before, >= 3 s of log)")
print("  by family:", dict(collections.Counter(e["family"] for e in crashes)))

## 1. Does the controller do what the planner asks?

In [ ]:
agree_b = agree_t = n_all = 0
ov_rows = collections.Counter()
for rid, d in PC.items():
    m = np.arange(d["n"]) >= 40                       # first 2 s: CARLA holds the brake
    thr, brk = pce.expected_control(d["speed"], d["tss"])
    ov = d["force_move"] | d["stop_sign"] | d["emergency_stop"]
    lb = d["brk"] > 0.5
    agree_b += int(((brk == lb) & m & ~ov).sum()); agree_t += int(((np.abs(thr - d["thr"]) < 0.02) & m & ~ov).sum())
    n_all += int((m & ~ov).sum())
    ov_rows["creep (stuck recovery)"] += int((d["force_move"] & ~d["emergency_stop"] & m).sum())
    ov_rows["stuck mode, creep blocked by safety box (car held)"] += int((d["emergency_stop"] & m).sum())
    ov_rows["stop-sign brake"] += int((d["stop_sign"] & m).sum())
total = sum(int((np.arange(d["n"]) >= 40).sum()) for d in PC.values())
table([{"check": "brake on/off matches the rule", "steps": n_all, "agreement": agree_b / n_all},
       {"check": "throttle within 0.02 of the rule", "steps": n_all, "agreement": agree_t / n_all}], fmt={"agreement": lambda v: f"{100 * v:.2f}%"})
table([{"override": k, "steps": v, "share of driving": v / total} for k, v in ov_rows.items()], fmt={"share of driving": lambda v: f"{100 * v:.2f}%"})
crash_ov = sum(bool((PC[e["route"]]["force_move"][e["step"] - 60:e["step"]] | PC[e["route"]]["stop_sign"][e["step"] - 60:e["step"]]).any()) for e in crashes)
print(f"collisions with an override active in the 3 s before impact: {crash_ov} of {len(crashes)}")

## 2. Speed tracking: target speed vs actual speed

Positive values mean the car is slower than TF++ wants. Because the controller limits acceleration, a gap right after the target speed jumps is normal. The question is whether crashes show anything unusual.

In [ ]:
def windows(sel, key, end_off=0, w=40):
    return np.concatenate([PC[x["route"]][key][x["step"] + end_off - w:x["step"] + end_off] for x in sel])
safe = collections.defaultdict(list)
for a in ew.maneuver_anchors(routes, EW, events):
    safe[a["label"]].append(a)
groups = {"crashes (3 s to 1 s before)": (crashes, -20), "successful junction turns": (safe["junction turn"], 0),
          "successful bypasses": (safe["obstacle bypass"], 0), "normal driving": (safe["normal driving"], 0)}
rows = []
for name, (sel, off) in groups.items():
    gap = windows(sel, "tss", off) - windows(sel, "speed", off)
    rows.append({"windows": name, "median target - actual (m/s)": float(np.median(gap)),
                 "car faster than target by > 1 m/s": float(np.mean(gap < -1)), "car slower than target by > 2 m/s": float(np.mean(gap > 2))})
table(rows, fmt={"car faster than target by > 1 m/s": pct, "car slower than target by > 2 m/s": pct})

tt = np.arange(-60, 1) * ew.DT
fig, ax = plt.subplots(figsize=(7, 3.2))
for key, c, lab in (("tss", "#c0392b", "target speed"), ("speed", "#2471a3", "actual speed")):
    A = np.array([PC[e["route"]][key][e["step"] - 60:e["step"] + 1] for e in crashes])
    ax.plot(tt, A.mean(0), color=c, label=lab)
A = np.array([PC[e["route"]]["brk"][e["step"] - 60:e["step"] + 1] for e in crashes])
ax2 = ax.twinx(); ax2.plot(tt, A.mean(0), color="#7f8c8d", ls=":", label="share braking"); ax2.set_ylim(0, 1)
ax.set_xlabel("seconds before impact"); ax.set_ylabel("m/s (mean over crashes)"); ax2.set_ylabel("share braking")
ax.legend(loc="upper left", fontsize=7); ax2.legend(loc="upper right", fontsize=7); ax.set_title(f"All {len(crashes)} collisions")
plt.tight_layout(); plt.savefig(FIG / "D2_speed_tracking.png", bbox_inches="tight"); plt.show()

## 3. Path tracking: did the car go where it planned?

For every step, the car's true position 0.5 s and 1 s later is measured against the path TF++ planned at that step. The **cross-track error** is the sideways distance from that path. Steps where the car barely moves are skipped.

In [ ]:
CT = {rid: (pce.cross_track(d, 10), pce.cross_track(d, 20)) for rid, d in PC.items()}
def ct_windows(sel, h, end_off=0, w=40):
    return np.concatenate([CT[x["route"]][h][x["step"] + end_off - w:x["step"] + end_off] for x in sel])
rows = []
for name, (sel, off) in groups.items():
    for h, lab in ((0, "0.5 s ahead"), (1, "1 s ahead")):
        v = ct_windows(sel, h, off); v = v[~np.isnan(v)]
        rows.append({"windows": name, "horizon": lab, "median (m)": float(np.median(v)), "90th pct (m)": float(np.percentile(v, 90)),
                     "share > 0.5 m": float(np.mean(v > 0.5))})
table(rows, fmt={"share > 0.5 m": pct})
allv = np.concatenate([CT[r][0][PC[r]["speed"] > 2] for r in PC]); allv = allv[~np.isnan(allv)]
fig, ax = plt.subplots(figsize=(6, 3))
for name, (sel, off), c in zip(groups, groups.values(), ["#c0392b", "#2471a3", "#b9770e", "#7f8c8d"]):
    v = np.sort(ct_windows(sel, 0, off)); v = v[~np.isnan(v)]
    ax.plot(v, np.arange(len(v)) / len(v), color=c, label=name)
ax.set_xlim(0, 1.0); ax.set_xlabel("cross-track error 0.5 s ahead (m)"); ax.set_ylabel("cumulative share"); ax.legend(fontsize=7)
ax.set_title("Did the car follow its own planned path?")
plt.tight_layout(); plt.savefig(FIG / "D3_path_tracking.png", bbox_inches="tight"); plt.show()
print(f"all moving steps: median {np.median(allv):.3f} m, 99th pct {np.percentile(allv, 99):.2f} m")

## 4. Braking response

Each time the brake switches on while the car is moving above 3 m/s, measure how much the speed drops over the next 0.5 s and 1 s, and the time until deceleration reaches −3 m/s². Rule-based brake decisions take effect immediately by design, so any delay here is the physics of the car, not the controller.

In [ ]:
def brake_events(rid, lo=0, hi=None):
    d = PC[rid]; b = d["brk"] > 0.5; hi = d["n"] if hi is None else hi
    on = np.where(b[1:] & ~b[:-1])[0] + 1
    out = []
    for i in on:
        if i < max(lo, 40) or i + 20 >= hi or d["speed"][i] < 3:
            continue
        acc = d["acc"][i:i + 20]
        reach = np.where(acc < -3)[0]
        out.append((d["speed"][i] - d["speed"][i + 10], d["speed"][i] - d["speed"][i + 20], reach[0] * ew.DT if len(reach) else np.nan))
    return out
crash_routes = {e["route"] for e in crashes}
normal = [x for r in PC if r not in crash_routes for x in brake_events(r)]
pre = [x for e in crashes for x in brake_events(e["route"], e["step"] - 60, e["step"])]
rows = []
for name, v in (("routes without a collision", normal), ("3 s before a collision", pre)):
    v = np.array(v)
    rows.append({"brake onsets": name, "n": len(v), "median speed drop in 0.5 s (m/s)": float(np.median(v[:, 0])),
                 "median speed drop in 1 s (m/s)": float(np.median(v[:, 1])), "median time to -3 m/s2 (s)": float(np.nanmedian(v[:, 2]))})
table(rows)

## 5. Steering behaviour

**Steering activity** is the total change in the steering command per second. A controller that oscillates or fights the plan would show high activity or saturation.

In [ ]:
rows = []
for name, (sel, off) in groups.items():
    st = np.concatenate([PC[x["route"]]["steer"][x["step"] + off - 40:x["step"] + off] for x in sel])
    act = [np.abs(np.diff(PC[x["route"]]["steer"][x["step"] + off - 41:x["step"] + off])).sum() / 2.0 for x in sel]
    rows.append({"windows": name, "steering saturated (|steer| > 0.95)": float(np.mean(np.abs(st) > 0.95)),
                 "median |steer|": float(np.median(np.abs(st))),
                 "median steering activity (total change per second)": float(np.median(act))})
table(rows, fmt={"steering saturated (|steer| > 0.95)": pct})

## 6. One line per collision

This is the control evidence used by Phase E (`results/bench2drive_v2/pce/control_per_collision.csv`). The window is the last 2 s before impact.

In [ ]:
rows = []
for e in crashes:
    d = PC[e["route"]]; s = e["step"]; w = slice(s - 40, s - 2)
    thr, brk = pce.expected_control(d["speed"][w], d["tss"][w])
    ov = (d["force_move"][w] | d["stop_sign"][w] | d["emergency_stop"][w])
    ct = CT[e["route"]][0][s - 40:s - 12]   # position 0.5 s later must still be before impact
    rows.append({"route": e["route"], "scenario": e["scenario"], "situation": e["situation"],
                 "max cross-track 0.5 s ahead (m)": round(float(np.nanmax(ct)), 2) if np.isfinite(ct).any() else "",
                 "median target - actual speed (m/s)": round(float(np.median(d["tss"][w] - d["speed"][w])), 2),
                 "brake matches rule": round(float(np.mean(brk == (d["brk"][w] > 0.5))), 2),
                 "override active": bool(ov.any()), "braking at impact": bool(d["brk"][s - 2] > 0.5)})
save_csv("control_per_collision", rows)
v = np.array([r["max cross-track 0.5 s ahead (m)"] for r in rows if r["max cross-track 0.5 s ahead (m)"] != ""])
print(f"collisions where the car stayed within 0.5 m of its plan for the last 2 s: {np.mean(v <= 0.5):.0%}  (within 1 m: {np.mean(v <= 1.0):.0%})")
print(f"collisions where throttle/brake matched the rule in every step of the last 2 s: {np.mean([r['brake matches rule'] == 1.0 for r in rows]):.0%}")
table(rows)

## Takeaways

1. **The longitudinal controller does exactly what the planner asks.** Re-running TF++'s throttle/brake rule on the logged target speed and speed reproduces the logged brake and throttle on **100.00%** of the 123,780 steps without an override.
2. **Overrides are rare around crashes.** The creep forward when stuck is active 1.2% of the time, and the stop-sign brake 0.18%. The "stuck, creep blocked by the safety box" state covers 24% of driving, but that is the car being held still while waiting. 7 of 61 collisions had an override active in the 3 s before impact.
3. **Speed: the car lags *behind* the target, not ahead of it.** In the 3 to 1 s before a crash, the car is more than 2 m/s slower than its target 44% of the time, against 8% in normal driving. It is accelerating as hard as allowed toward a target TF++ keeps high. Avoiding these crashes needed a lower target speed, not better speed control.
4. **The car follows its own path closely.**

   | Cross-track error 0.5 s ahead | Median | 90th percentile |
   |---|---|---|
   | Crash windows | 0.02 m | 0.23 m |
   | Normal driving | 0.01 m | 0.09 m |

   Over the last 2 s before impact, 88% of collisions stayed within 1 m of their plan. The exceptions are high-speed lane changes and one fast bypass (1.0–2.7 m). Successful high-speed lane changes show the same kind of lag (3.9% of steps more than 1 m off), so this is a controller limitation, not a malfunction.
5. **Braking and steering behave normally.** When the brake engages, speed drops 1.50 m/s in 0.5 s before crashes against 1.45 m/s elsewhere, and deceleration reaches −3 m/s² after 0.15 s in both cases. There is no steering saturation or oscillation: activity is 0.15 per second, between normal driving (0.09) and successful turns (0.31).

**Control is not a dominant failure source.** The controller faithfully carries out the plan, including unsafe ones. The only control-related weakness is lag during high-speed lane changes.
